# FER2013 Transfer Learning Benchmark — VGG16, VGG19, ResNet18

**Run on Kaggle Notebook (GPU accelerator recommended)**

Sections:
1. Install dependencies
2. Reproducibility — global seed (42)
3. Download FER2013 via `kagglehub`
4. Data utilities (inlined from `load_data.py`)
5. Preprocessing utilities (inlined from `data_preprocessing.py`)
6. Core imports
7. Load dataset
8. Visualize class distribution
9. Compute class weights
10. Hyperparameter options (project baseline + alternatives)
11. Model builders (VGG16 / VGG19 / ResNet18)
12. Multi-model training (batch=64, epochs=50, lr=3e-4)
13. Comparative evaluation:
- Accuracy/Loss curves for all models
- Confusion matrices on train and test for each model

## 1. Install Dependencies

In [ ]:
%pip install -q kagglehub keras-cv

## 2. Reproducibility — Global Seed (42)

Setting seeds across Python, NumPy, and TensorFlow so every run is deterministic and results can be fairly compared against other approaches.

In [ ]:
import os, random
import numpy as np
import tensorflow as tf

SEED = 42

os.environ["PYTHONHASHSEED"]       = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"All seeds set to {SEED}")
print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## 3. Download FER2013 via `kagglehub`

This follows the same pattern as `emotion_recognition_colab.ipynb`:
- download `msambare/fer2013` with `kagglehub.dataset_download(...)`
- use the returned folder as `DATASET_ROOT`
- set `TRAIN_DIR` and `TEST_DIR` from that root.

In [ ]:
import os
import kagglehub

# Download FER2013 dataset (folder-structured image version).
DATASET_ROOT = kagglehub.dataset_download("msambare/fer2013")

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

print("Dataset downloaded to:", DATASET_ROOT)
print("Train dir:", TRAIN_DIR)
print("Test  dir:", TEST_DIR)

if not os.path.isdir(TRAIN_DIR) or not os.path.isdir(TEST_DIR):
    raise RuntimeError("Downloaded dataset does not contain expected 'train/' and 'test/' folders.")

In [ ]:
import glob

train_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
test_classes = sorted([d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))])

print("Train classes:", train_classes)
print("Test  classes:", test_classes)

print("\nSample count by class:")
for cls in train_classes:
    train_count = len(glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg")))
    test_count = len(glob.glob(os.path.join(TEST_DIR, cls, "*.jpg")))
    print(f"{cls:<10} train={train_count:5d} | test={test_count:5d}")

## 4. Data Utilities

*(inlined from `load_data.py`; uses `TRAIN_DIR` and `TEST_DIR` from Kaggle path setup above)*

In [ ]:
import glob
from PIL import Image

# ---------------------------------------------------------------------------
# Emotion labels — sorted alphabetically for consistent integer encoding
# ---------------------------------------------------------------------------
LABELS       = sorted(["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"])
LABEL_TO_IDX = {label: idx for idx, label in enumerate(LABELS)}


def _load_split(split_dir: str):
    """
    Load all images from a split directory (train or test).

    Directory structure expected:
        split_dir/
            {emotion_class}/
                *.jpg

    Returns:
        X : np.ndarray of shape (N, 48, 48, 1), dtype float32, values in [0, 1]
        y : np.ndarray of shape (N, num_classes), dtype float32 — one-hot encoded labels
    """
    images = []
    labels = []

    for label in LABELS:
        class_dir = os.path.join(split_dir, label)
        if not os.path.isdir(class_dir):
            continue
        for img_path in glob.glob(os.path.join(class_dir, "*.jpg")):
            img = Image.open(img_path).convert("L")           # L = grayscale (monochrome)
            arr = np.array(img, dtype=np.float32) / 255.0     # normalise to [0, 1]
            images.append(arr)
            labels.append(LABEL_TO_IDX[label])

    X     = np.array(images, dtype=np.float32)[..., np.newaxis]  # (N, 48, 48, 1)
    y_idx = np.array(labels,  dtype=np.int64)                     # (N,)
    y     = np.eye(len(LABELS), dtype=np.float32)[y_idx]          # (N, num_classes) one-hot
    return X, y


def load_training_dataset(train_dir: str = TRAIN_DIR):
    """
    Load the training split.

    Returns:
        X : np.ndarray  shape (N, 48, 48, 1)  — pixel values in [0, 1]
        y : np.ndarray  shape (N, num_classes) — one-hot encoded labels
    """
    return _load_split(train_dir)


def load_testing_dataset(test_dir: str = TEST_DIR):
    """
    Load the test split.

    Returns:
        X : np.ndarray  shape (N, 48, 48, 1)  — pixel values in [0, 1]
        y : np.ndarray  shape (N, num_classes) — one-hot encoded labels
    """
    return _load_split(test_dir)


print("LABELS      :", LABELS)
print("LABEL_TO_IDX:", LABEL_TO_IDX)

## 5. Preprocessing Utilities

*(inlined from `data_preprocessing.py`)*

In [ ]:
from matplotlib.patches import Rectangle
from sklearn.utils.class_weight import compute_class_weight


def plot_class_distribution(y: np.ndarray, title: str = "Emotion Class Distribution") -> None:
    """
    Plot a bar chart showing the number of samples per emotion class.

    Args:
        y     : np.ndarray of shape (N, num_classes) — one-hot encoded labels
        title : title of the plot
    """
    counts = y.sum(axis=0).astype(int)   # (num_classes,)

    colors = ["#e74c3c", "#8e44ad", "#2980b9", "#f1c40f",
              "#95a5a6", "#2ecc71", "#e67e22"]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(LABELS, counts, color=colors, edgecolor="black", linewidth=0.8)

    for bar, count in zip(bars, counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(counts) * 0.01,
            str(count),
            ha="center", va="bottom", fontsize=11, fontweight="bold",
        )

    legend_handles = [
        Rectangle((0, 0), 1, 1, color=colors[i], edgecolor="black", linewidth=0.8)
        for i in range(len(LABELS))
    ]
    ax.legend(legend_handles, LABELS, title="Emotions", loc="upper right", fontsize=10)

    ax.set_title(title, fontsize=16, fontweight="bold", pad=15)
    ax.set_xlabel("Emotion", fontsize=13)
    ax.set_ylabel("Number of Samples", fontsize=13)
    ax.set_ylim(0, max(counts) * 1.15)
    ax.tick_params(axis="x", labelsize=11)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()


def get_class_weights(y: np.ndarray, max_weight: float):
    """
    Return balanced class weights, capped at max_weight.

    Args:
        y          : np.ndarray of shape (N, num_classes) — one-hot encoded labels
        max_weight : upper cap applied to every weight

    Returns:
        dict {class_idx: weight}
    """
    y_integers = np.argmax(y, axis=1)   # (N,) integer labels
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_integers),
        y=y_integers,
    )
    return {idx: float(np.minimum(w, max_weight)) for idx, w in enumerate(weights)}


print("Preprocessing utilities ready.")

## 6. Core Imports

In [ ]:
import keras_cv
from tensorflow.keras.applications import VGG16, VGG19
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess_input
from tensorflow.keras.applications.vgg19 import preprocess_input as vgg19_preprocess_input
import matplotlib.pyplot as plt

print("Core imports OK.")

## 7. Load Dataset

In [ ]:
X_train, y_train = load_training_dataset(TRAIN_DIR)
X_test,  y_test  = load_testing_dataset(TEST_DIR)

print(f"Train  →  X: {X_train.shape}   y: {y_train.shape}")
print(f"Test   →  X: {X_test.shape}    y: {y_test.shape}")

## 8. Visualize Class Distribution

In [ ]:
plot_class_distribution(y_train, title="FER2013 — Training Set Class Distribution")
plot_class_distribution(y_test, title="FER2013 — Test Set Class Distribution")

## 9. Class Weights

Computed via `sklearn`'s *balanced* strategy to compensate for the class imbalance visible above.

In [ ]:
MAX_WEIGHT = 4.0   # aligned with project script defaults for stronger imbalance correction

class_weights = get_class_weights(y_train, max_weight=MAX_WEIGHT)

print(f"{'Class':<12} {'Index':>5}  {'Weight':>8}")
print("-" * 30)
for label, idx in LABEL_TO_IDX.items():
    print(f"{label:<12} {idx:>5}  {class_weights[idx]:>8.4f}")

In [ ]:
# Visual overview of class weights
fig, ax = plt.subplots(figsize=(9, 4))

weight_values = [class_weights[LABEL_TO_IDX[l]] for l in LABELS]
colors = ["#e74c3c", "#8e44ad", "#2980b9", "#f1c40f", "#95a5a6", "#2ecc71", "#e67e22"]
bars = ax.bar(LABELS, weight_values, color=colors, edgecolor="black", linewidth=0.8)

for bar, w in zip(bars, weight_values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{w:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.axhline(1.0, color="grey", linestyle="--", linewidth=1, label="weight = 1 (balanced)")
ax.set_title("Class Weights (balanced, capped at {:.1f})".format(MAX_WEIGHT),
             fontsize=14, fontweight="bold")
ax.set_xlabel("Emotion", fontsize=12)
ax.set_ylabel("Weight", fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.45)
plt.tight_layout()
plt.show()

## 10. Hyperparameter Options (same structure, pretrained-focused)

We keep your required core setup as default:
- `batch_size = 64`
- `epochs = 50`
- `learning_rate = 0.0003`

Pretrained policy for this notebook:
- Use pretrained TensorFlow ecosystem backbones for all three models.
- `VGG16` and `VGG19` come from `tf.keras.applications` (ImageNet).
- `ResNet18` comes from a KerasCV pretrained preset (ImageNet).

This keeps your three-model benchmark fully pretrained while preserving a readable Sequential training flow.

## 11. Model & Training Imports

All imports required for multi-model building, scheduling, training, and evaluation.

In [ ]:
# ── Keras / TensorFlow ────────────────────────────────────────────────────────
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Lambda, Dense, BatchNormalization, Dropout, LeakyReLU,
    Resizing, RandomZoom, RandomTranslation, RandomRotation, RandomFlip,
    GaussianNoise, GlobalAveragePooling2D,
    )
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ── Evaluation / Visualization ───────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, f1_score as sklearn_f1_score
import seaborn as sns

print("Multi-model imports ready.")

## 12. Define Pretrained Backbone & Fine-Tuned Model

Same pattern as your previous notebook style:
- `create_pretrained_model(model_name)`
- `create_fine_tuned_model(model_name, cfg)`

Implementation style is Sequential-first for readability, consistent with your project files.

In [ ]:
# ── Fixed requirements from project request ───────────────────────────────────
FIXED_BATCH_SIZE = 64
FIXED_EPOCHS = 50
FIXED_LR = 3e-4

# ── Candidate recipes (from repo baseline + common transfer-learning practice)
HP_OPTIONS = {
    "repo_baseline_plateau": {
        "scheduler": "plateau",
        "label_smoothing": 0.10,
        "dropout": 0.50,
        "l2": 1e-3,
        "max_class_weight": 4.0,
        "early_stopping_patience": 12,
        "lr_factor": 0.5,
        "lr_patience": 4,
        "min_lr": 1e-6,
    },
    "conservative_plateau": {
        "scheduler": "plateau",
        "label_smoothing": 0.05,
        "dropout": 0.40,
        "l2": 5e-4,
        "max_class_weight": 4.0,
        "early_stopping_patience": 15,
        "lr_factor": 0.5,
        "lr_patience": 5,
        "min_lr": 1e-6,
    },
    "cosine_restart": {
        "scheduler": "cosine_restart",
        "label_smoothing": 0.10,
        "dropout": 0.50,
        "l2": 1e-3,
        "max_class_weight": 4.0,
        "early_stopping_patience": 12,
        "first_decay_epochs": 5,
        "t_mul": 2.0,
        "m_mul": 0.9,
        "alpha": 1e-2,
    },
}

# Select default recipe
SELECTED_RECIPE = "repo_baseline_plateau"
cfg = dict(HP_OPTIONS[SELECTED_RECIPE])
cfg["batch_size"] = FIXED_BATCH_SIZE
cfg["epochs"] = FIXED_EPOCHS
cfg["lr"] = FIXED_LR

# Recompute class weights with selected recipe cap
class_weights = get_class_weights(y_train, max_weight=cfg["max_class_weight"])

# Shared augmentation settings
AUG_ZOOM = 0.03
AUG_TRANSLATE = 0.03
AUG_ROTATE = 0.03

def apply_common_augmentation(x, prefix: str):
    """Apply a light, FER-safe augmentation stack."""
    x = RandomZoom(height_factor=(-AUG_ZOOM, AUG_ZOOM), name=f"{prefix}_zoom")(x)
    x = RandomTranslation(
        height_factor=(-AUG_TRANSLATE, AUG_TRANSLATE),
        width_factor=(-AUG_TRANSLATE, AUG_TRANSLATE),
        name=f"{prefix}_translate",
    )(x)
    x = RandomRotation(factor=(-AUG_ROTATE, AUG_ROTATE), name=f"{prefix}_rotate")(x)
    x = RandomFlip("horizontal", name=f"{prefix}_flip")(x)
    x = GaussianNoise(stddev=0.03, name=f"{prefix}_noise")(x)
    return x

def build_callbacks(cfg):
    callbacks = []
    callbacks.append(
        EarlyStopping(
            monitor="val_loss",
            patience=cfg["early_stopping_patience"],
            restore_best_weights=True,
            verbose=1,
        )
    )
    if cfg["scheduler"] == "plateau":
        callbacks.append(
            ReduceLROnPlateau(
                monitor="val_loss",
                factor=cfg["lr_factor"],
                patience=cfg["lr_patience"],
                min_lr=cfg["min_lr"],
                verbose=1,
            )
        )
    return callbacks

def build_optimizer(cfg, steps_per_epoch: int):
    if cfg["scheduler"] == "cosine_restart":
        first_decay_steps = max(1, int(cfg["first_decay_epochs"] * steps_per_epoch))
        lr_schedule = CosineDecayRestarts(
            initial_learning_rate=cfg["lr"],
            first_decay_steps=first_decay_steps,
            t_mul=cfg["t_mul"],
            m_mul=cfg["m_mul"],
            alpha=cfg["alpha"],
        )
        return Adam(learning_rate=lr_schedule)
    return Adam(learning_rate=cfg["lr"])

print("Selected recipe:", SELECTED_RECIPE)
print(f"batch_size={cfg['batch_size']}, epochs={cfg['epochs']}, lr={cfg['lr']}")
print("scheduler:", cfg["scheduler"])

In [ ]:
NUM_CLASSES = len(LABELS)

RESNET18_PRESET_CANDIDATES = [
    "resnet18_imagenet",
    "resnet_18_imagenet",
    "resnet18",
]

BACKBONE_CONFIG = {
    "vgg16": {
        "name": "VGG16",
        "preprocess": vgg16_preprocess_input,
    },
    "vgg19": {
        "name": "VGG19",
        "preprocess": vgg19_preprocess_input,
    },
    "resnet18": {
        "name": "ResNet18",
        "preprocess": None,
    },
}

def _load_kerascv_resnet18_backbone():
    """Try multiple preset aliases to load a pretrained ResNet18 backbone."""
    last_error = None
    for preset in RESNET18_PRESET_CANDIDATES:
        try:
            backbone = keras_cv.models.ResNet18Backbone.from_preset(preset)
            print(f"Loaded KerasCV ResNet18 preset: {preset}")
            return backbone
        except Exception as e:
            last_error = e
    raise RuntimeError(
        "Could not load a pretrained ResNet18 preset from KerasCV. "
        "Try updating keras-cv (%pip install -U keras-cv) and rerun this cell."
    ) from last_error

def create_pretrained_model(model_name: str):
    """Load pretrained backbone according to model_name."""
    model_name = model_name.lower()
    if model_name not in BACKBONE_CONFIG:
        raise ValueError(f"Unsupported model name: {model_name}")

    if model_name == "vgg16":
        base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
        base_model.trainable = False
        for layer in base_model.layers:
            if layer.name.startswith("block5"):
                layer.trainable = True
        preprocess_fn = BACKBONE_CONFIG[model_name]["preprocess"]
        return base_model, preprocess_fn, "feature_map"

    if model_name == "vgg19":
        base_model = VGG19(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
        base_model.trainable = False
        for layer in base_model.layers:
            if layer.name.startswith("block5"):
                layer.trainable = True
        preprocess_fn = BACKBONE_CONFIG[model_name]["preprocess"]
        return base_model, preprocess_fn, "feature_map"

    # ResNet18 path via KerasCV pretrained backbone presets.
    base_model = _load_kerascv_resnet18_backbone()
    base_model.trainable = False
    for layer in base_model.layers:
        lname = layer.name.lower()
        if "stack4" in lname or "stage4" in lname or "conv5" in lname:
            layer.trainable = True
    return base_model, None, "feature_map"

def create_fine_tuned_model(model_name: str, cfg):
    """Build Sequential fine-tuning pipeline in the same style as project notebooks."""
    base_model, preprocess_fn, output_mode = create_pretrained_model(model_name)
    model_name = model_name.lower()

    layers_list = [
        Input(shape=(48, 48, 1)),
        Lambda(lambda x: tf.repeat(x, repeats=3, axis=-1)),
        Resizing(224, 224, interpolation="bilinear"),
        RandomZoom(height_factor=(-AUG_ZOOM, AUG_ZOOM)),
        RandomTranslation(height_factor=(-AUG_TRANSLATE, AUG_TRANSLATE),
                          width_factor=(-AUG_TRANSLATE, AUG_TRANSLATE)),
        RandomRotation(factor=(-AUG_ROTATE, AUG_ROTATE)),
        RandomFlip("horizontal"),
        GaussianNoise(stddev=0.03),
    ]

    if preprocess_fn is not None:
        layers_list.append(Lambda(lambda x: preprocess_fn(x * 255.0)))
    elif model_name == "resnet18":
        # KerasCV backbone presets are trained on ImageNet-scale RGB inputs.
        layers_list.append(Lambda(lambda x: x * 255.0))

    layers_list.append(base_model)

    if output_mode == "feature_map":
        layers_list.append(GlobalAveragePooling2D())

    layers_list.extend([
        Dense(512, kernel_regularizer=l2(cfg["l2"])),
        BatchNormalization(),
        LeakyReLU(negative_slope=0.01),
        Dropout(cfg["dropout"]),
        Dense(NUM_CLASSES, activation="softmax"),
    ])

    model = Sequential(layers_list, name=BACKBONE_CONFIG[model_name]["name"].upper())
    return model

def train_single_model(model_name: str, cfg):
    """Compile and train one model; returns (model, history)."""
    tf.keras.backend.clear_session()
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model = create_fine_tuned_model(model_name, cfg)
    steps_per_epoch = int(np.ceil(X_train.shape[0] / cfg["batch_size"]))
    optimizer = build_optimizer(cfg, steps_per_epoch)

    model.compile(
        optimizer=optimizer,
        loss=CategoricalCrossentropy(label_smoothing=cfg["label_smoothing"]),
        metrics=["accuracy"],
    )

    print(f"\n--- Training {model.name} ---")
    history = model.fit(
        x=X_train,
        y=y_train,
        batch_size=cfg["batch_size"],
        epochs=cfg["epochs"],
        class_weight=class_weights,
        validation_data=(X_test, y_test),
        shuffle=True,
        callbacks=build_callbacks(cfg),
        verbose=1,
    )

    return model, history

print("Pretrained Sequential builders ready: VGG16, VGG19, ResNet18")

## 13. Multi-Model Training

Train all requested models with shared core settings:
- `batch_size = 64`
- `epochs = 50`
- `learning_rate = 0.0003`

Models: `VGG16` (ImageNet), `VGG19` (ImageNet), `ResNet18` (KerasCV pretrained preset).

In [ ]:
MODEL_ORDER = ["vgg16", "vgg19", "resnet18"]
benchmark_results = {}

for model_name in MODEL_ORDER:
    model, history_obj = train_single_model(model_name, cfg)
    best_epoch = int(np.argmax(history_obj.history["val_accuracy"])) + 1
    best_val_acc = float(np.max(history_obj.history["val_accuracy"]))

    benchmark_results[model_name] = {
        "model": model,
        "history": history_obj.history,
        "best_epoch": best_epoch,
        "best_val_accuracy": best_val_acc,
    }

    print(
        f"[{model_name.upper()}] best val_accuracy = {best_val_acc:.4f} at epoch {best_epoch}"
    )

print("\nAll models finished training.")

## 14. Comparative Evaluation

This section generates:
1. Train-set confusion matrices for VGG16, VGG19, ResNet18
2. Test-set confusion matrices for VGG16, VGG19, ResNet18
3. Comparative training curves (accuracy and loss)

In [ ]:
def plot_confusion_matrices(results, X_eval, y_eval, split_name: str, pred_batch_size: int = 256):
    """Plot one confusion matrix per model for a given split."""
    y_true = np.argmax(y_eval, axis=1)

    fig, axes = plt.subplots(1, len(results), figsize=(7 * len(results), 6))
    if len(results) == 1:
        axes = [axes]

    for ax, (model_name, pack) in zip(axes, results.items()):
        model = pack["model"]
        y_prob = model.predict(X_eval, batch_size=pred_batch_size, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)

        cm = confusion_matrix(y_true, y_pred)
        acc = np.mean(y_true == y_pred) * 100
        f1 = sklearn_f1_score(y_true, y_pred, average="weighted") * 100

        sns.heatmap(
            cm,
            ax=ax,
            annot=True,
            fmt="d",
            cmap="YlGnBu",
            cbar=False,
            xticklabels=LABELS,
            yticklabels=LABELS,
        )
        ax.set_title(f"{model_name.upper()} — {split_name}\nAcc: {acc:.2f}% | F1: {f1:.2f}%",
                     fontsize=11, fontweight="bold")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")

    plt.tight_layout()
    plt.show()

# Train confusion matrices
plot_confusion_matrices(benchmark_results, X_train, y_train, split_name="Training")

# Test confusion matrices
plot_confusion_matrices(benchmark_results, X_test, y_test, split_name="Testing")

In [ ]:
def plot_comparative_curves(results):
    """Expressive side-by-side comparison of train/validation accuracy and loss."""
    sns.set_style("whitegrid")
    fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(18, 6))
    palette = sns.color_palette("tab10", n_colors=len(results))

    for color, (model_name, pack) in zip(palette, results.items()):
        hist = pack["history"]
        epochs = range(1, len(hist["accuracy"]) + 1)

        # Accuracy curves
        ax_acc.plot(epochs, hist["accuracy"], color=color, linewidth=2.2,
                    label=f"{model_name.upper()} Train")
        ax_acc.plot(epochs, hist["val_accuracy"], color=color, linewidth=2.2,
                    linestyle="--", label=f"{model_name.upper()} Val")

        # Loss curves
        ax_loss.plot(epochs, hist["loss"], color=color, linewidth=2.2,
                     label=f"{model_name.upper()} Train")
        ax_loss.plot(epochs, hist["val_loss"], color=color, linewidth=2.2,
                     linestyle="--", label=f"{model_name.upper()} Val")

        # Highlight best val accuracy point
        best_epoch = pack["best_epoch"]
        best_acc = pack["best_val_accuracy"]
        ax_acc.scatter(best_epoch, best_acc, color=color, s=60, zorder=5)

    ax_acc.set_title("Model Comparison — Accuracy", fontsize=14, fontweight="bold")
    ax_acc.set_xlabel("Epoch")
    ax_acc.set_ylabel("Accuracy")
    ax_acc.legend(fontsize=9, ncol=2)
    ax_acc.grid(True, linestyle="--", alpha=0.4)

    ax_loss.set_title("Model Comparison — Loss", fontsize=14, fontweight="bold")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.legend(fontsize=9, ncol=2)
    ax_loss.grid(True, linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

plot_comparative_curves(benchmark_results)